# TsekTxt – retrain all three models (Colab)
**Before running:** Runtime → Change runtime type → **T4 GPU** (or better). Add a Colab *Secret* (key icon on the left) named `HF_TOKEN` = a Hugging Face token with **Write** access, and turn on "Notebook access".
The GitHub repo must already contain the NEW code and `data/raw/combined_dataset_clean.csv` (push them first). Run the cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch; print('CUDA available:', torch.cuda.is_available())

## 2. Get the code (repo is public, so no GitHub token is needed)

In [ ]:
import os
os.chdir('/content')
!rm -rf tsektxt-model-training
!git clone https://github.com/hansjio/tsektxt-model-training.git
%cd tsektxt-model-training
!pip install -q -r requirements.txt

## 3. Log in to Hugging Face using the Colab secret

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import whoami
print('Logged in as:', whoami()['name'])   # must match hub.user in configs/config.yaml

## 4. Clean the data and make the 70/15/15 split
The cleaned CSV must be in the repo at `data/raw/combined_dataset_clean.csv`. If not, upload it with the Files panel on the left (drag into `data/raw/`).

In [ ]:
!ls -la data/raw
!python src/clean_data.py --input data/raw/combined_dataset_clean.csv
!python src/make_splits.py

## 5. Trial run first (one model, NOT uploaded). Check it finishes and prints metrics.

In [ ]:
!python src/run_all.py --only xlmr

## 5b. Full run: trains all three, uploads each to Hugging Face, then compares (an hour or so on a T4 — an estimate, not measured)

In [ ]:
!python src/run_all.py --push

## 6. (Later) Evaluate on your Taglish election test set
Upload `taglish_test.csv` (columns `id,text,label`) into `data/external/`, then:

In [ ]:
for m in ['roberta-tagalog','mbert','xlmr']:
    !python src/evaluate_external.py --model {m} --csv data/external/taglish_test.csv
!python src/compare.py

## 7. Download the results so you can commit them to GitHub

In [ ]:
!zip -qr results.zip results data/processed/cleaning_report.json -x 'results/*/checkpoints/*'
from google.colab import files; files.download('results.zip')